# Exploratory Data Analysis for the Australian Weather Dataset
For this assignment, the exploratory data analysis is split from the rest of the tasks to make the code less connected. This is done after experience gained from the previous assignment that the EDA would be shuffled around the notebook depending on which other aspects of the task needed to be done in which order. This also makes it easier to work on the project in parallel and ensure that once the EDA is finished, it does not have to tested after every change in the main notebook. In summary, the EDA for the Australian Weather Dataset is decoupled from the main notebook to get the advantages of loosely coupled code.



In [ ]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import TargetEncoder, OrdinalEncoder

In [ ]:
eda_dataset = pd.read_csv('Dataset/weatherAUS.csv')
pd.DataFrame(eda_dataset)
eda_dataset.head()

In [ ]:
eda_dataset.info()

In [ ]:
eda_dataset.isna().sum()

Rows with missing values in the target value are dropped since they offer no value to supervised learning methods.

In [ ]:
eda_dataset = eda_dataset.dropna(subset=["RainTomorrow"])

eda_dataset.isna().sum()

Making a numerical dataset and a categorical dataset to make analysis easier

## Temporary Encoding
For certain parts of the analysis, such as the correlation heatmap, it is necessary to use encoded data. There is therefore a temporary encoding of the categorical and time series values connected to the dataset. The values are encoded in such a way that all parts of features are carried forward in the dataset, and that the encoding makes sense for the feature domain. For example, ordinal encoder is not used on location, since there is no inherent order (other than perhaps coordinate location) to them. Another example is that it may be that the year is unnecessary as a feature since historically rain is hard to predict based on yearly intervals. But, it is still included since this is the analysis part.


In [ ]:
# Create a new encoded version of the dataset which will be used for certain parts of the analysis.
encoded_eda_dataset = eda_dataset.copy()

In [ ]:
encoded_eda_dataset['RainTomorrow'] = encoded_eda_dataset['RainTomorrow'].replace({'No':0, 'Yes':1}).astype(float)

Encoding of categorical data

In [ ]:
# Target encoding of location
x = encoded_eda_dataset[['Location']]
y = encoded_eda_dataset['RainTomorrow']

encoder = TargetEncoder()
encoded_eda_dataset['Location'] = encoder.fit_transform(x,y)

Encoding of numerical data

In [ ]:
# Binary Encoding of RainToday
encoded_eda_dataset['RainToday'] = encoded_eda_dataset['RainToday'].replace({'No': 0, 'Yes': 1}).astype(float)

### Cyclical encoding

This dataset includes multiple features which have cyclical properties. This means that the order of the data is important, but there is a wrap-around of the order at the start and finish. The date includes seasonality, with certain patterns reoccurring depending on which time of year it is. For example, in Australia, rain is more common in the summer. The wind direction is also cyclical, since the direction loops around for every full loop. Ordinal encoding would not retain the wrap-around property that cyclical features contain. Cyclical encoding is used for these features instead.

The relevant features with cyclic properties are as follows:
- WindGustDir
- WindDir9am
- WindDir3am
- date: day
- date: month

A general function for encoding with cyclic data is created which takes a Pandas series object and its corresponding period length. The function returns the same pandas series, encoded for the given period. This encoding includes both a cosine and sine wave fitted version of the Pandas series. Both are needed to keep track of the cycle, since multiple values share the same sine and cosine values in the same cycle. The function only works for numerical data.

In [ ]:
# Function that takes a Pandas series and a corresponding period.
# Only pass numerical series.
# def cyclicEncoder(series, period):
#     # Filter for missing values. (Not needed for main notebook.)
#     # To prevent missing values from ruining the encoding, a check is made with the following requirement:
#     missing = series.isna()
#     # If values are missing, they are simply passed through.
#     sin = np.where(missing, np.nan, np.sin(2*np.pi*series/period))
#     cos = np.where(missing, np.nan, np.cos(2*np.pi*series/period))
#     return sin, cos

def cyclicEncoder(series, period):

    sin = np.sin(2*np.pi*series/period)
    cos = np.cos(2*np.pi*series/period)
    return sin, cos



#### Data Preparation

In [ ]:
# Ready date related data
# Convert to Pandas datetime format
encoded_eda_dataset["Date"] = pd.to_datetime(encoded_eda_dataset["Date"])
# Extract year, month and day based on year.
# Year is already "pre encoded" and can be left untouched.
encoded_eda_dataset["Year"] = encoded_eda_dataset["Date"].dt.year
encoded_eda_dataset["Month"] = encoded_eda_dataset["Date"].dt.month
encoded_eda_dataset["DayOfYear"] = encoded_eda_dataset["Date"].dt.day_of_year


In [ ]:
# Ready wind related data
# Find the unique entries in the wind columns
encoded_eda_dataset["WindGustDir"].unique()

For analytical purposes, the missing values are kept. The rest of the values are manually encoded using domain knowledge.

In [ ]:
encoded_eda_dataset['WindGustDir'] = encoded_eda_dataset['WindGustDir'].replace({'N': 16,
                                                                                 'NNE': 1,
                                                                                 'NE': 2,
                                                                                 'ENE': 3,
                                                                                 'E': 4,
                                                                                 'ESE': 5,
                                                                                 'SE': 6,
                                                                                 'SSE': 7,
                                                                                 'S': 8,
                                                                                 'SSW': 9,
                                                                                 'SW': 10,
                                                                                 'WSW': 11,
                                                                                 'W': 12,
                                                                                 'WNW': 13,
                                                                                 'NW': 14,
                                                                                 'NNW': 15,
                                                                                 }).astype(float)
# Floats has to be used for the nan objects to be translated to floats as well.
# Otherwise, the array with nans will be of dtype object which will cause the sin and cos
# functions from numpy to fail.

encoded_eda_dataset['WindDir9am'] = encoded_eda_dataset['WindDir9am'].replace({'N': 0,
                                                                                 'NNE': 1,
                                                                                 'NE': 2,
                                                                                 'ENE': 3,
                                                                                 'E': 4,
                                                                                 'ESE': 5,
                                                                                 'SE': 6,
                                                                                 'SSE': 7,
                                                                                 'S': 8,
                                                                                 'SSW': 9,
                                                                                 'SW': 10,
                                                                                 'WSW': 11,
                                                                                 'W': 12,
                                                                                 'WNW': 13,
                                                                                 'NW': 14,
                                                                                 'NNW': 15,
                                                                                 }).astype(float)

encoded_eda_dataset['WindDir3pm'] = encoded_eda_dataset['WindDir3pm'].replace({'N': 0,
                                                                                 'NNE': 1,
                                                                                 'NE': 2,
                                                                                 'ENE': 3,
                                                                                 'E': 4,
                                                                                 'ESE': 5,
                                                                                 'SE': 6,
                                                                                 'SSE': 7,
                                                                                 'S': 8,
                                                                                 'SSW': 9,
                                                                                 'SW': 10,
                                                                                 'WSW': 11,
                                                                                 'W': 12,
                                                                                 'WNW': 13,
                                                                                 'NW': 14,
                                                                                 'NNW': 15,
                                                                                 }).astype(float)
# Check for array datatype and encoded values.
encoded_eda_dataset['WindGustDir'].unique()

#### Use the Cyclic Encoding Function

In [ ]:
# For future use
# encoded_eda_dataset["Month_Sin"],encoded_eda_dataset["Month_Cos"]  = cyclicEncoder(encoded_eda_dataset["Month"], MONTH_PERIOD)
# encoded_eda_dataset["Day_Sin"],encoded_eda_dataset["Day_Cos"]  = cyclicEncoder(encoded_eda_dataset["DayOfYear"], encoded_eda_dataset['Days_in_year'])
# encoded_eda_dataset["WindGustDir_Sin"],encoded_eda_dataset["WindGustDir_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindGustDir"], WIND_DIR)
# encoded_eda_dataset["WindDir9am_Sin"],encoded_eda_dataset["WindDir9am_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindDir9am"], WIND_DIR)
# encoded_eda_dataset["WindDir3pm_Sin"],encoded_eda_dataset["WindDir3pm_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindDir3pm"], WIND_DIR)

In [ ]:
# Period lengths:
MONTH_PERIOD = 12
WIND_DIR = 16
# The day is more complicated since it varies based on the year.
# A new temporary pandas series is created and passed in the period based on the year
year = encoded_eda_dataset['Year']
isLeapYear = ((year % 4 == 0) & (year % 100 != 0)) | (year % 400 == 0)
encoded_eda_dataset['DaysInYear'] = np.select([isLeapYear], [366], default=365)


encoded_eda_dataset["Month_Sin"],encoded_eda_dataset["Month_Cos"]  = cyclicEncoder(encoded_eda_dataset["Month"], MONTH_PERIOD)
encoded_eda_dataset["Day_Sin"],encoded_eda_dataset["Day_Cos"]  = cyclicEncoder(encoded_eda_dataset["DayOfYear"], encoded_eda_dataset['DaysInYear'])
encoded_eda_dataset["WindGustDir_Sin"],encoded_eda_dataset["WindGustDir_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindGustDir"], WIND_DIR)
encoded_eda_dataset["WindDir9am_Sin"],encoded_eda_dataset["WindDir9am_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindDir9am"], WIND_DIR)
encoded_eda_dataset["WindDir3pm_Sin"],encoded_eda_dataset["WindDir3pm_Cos"]  = cyclicEncoder(encoded_eda_dataset["WindDir3pm"], WIND_DIR)

# Remove redundant columns
encoded_eda_dataset.drop(["Date", "DaysInYear", "Month", "DayOfYear", "WindGustDir", "WindDir9am", "WindDir3pm"], axis=1, inplace=True)

In [ ]:
encoded_eda_dataset["Month_Sin"].unique()
# encoded_eda_dataset["Month_Cos"].unique()
# encoded_eda_dataset["WindGustDir_Sin"].unique()

### Check encoding

In [ ]:
encoded_eda_dataset.head()

## Data Analysis

### Univariate Analysis

In [ ]:
# See how many different locations are used.
eda_dataset["Location"].nunique()

In [ ]:
# Check if every location has the same amount of observations
eda_dataset.value_counts("Location")

As can be seen from the result, the gap between the location with the most and fourth least entries is small, but not negligible. The gap is 3436 - 3001 = 435, which is an increase of about $\frac{435}{3001}$ $\cdot$ 100% $\approx$ 14.5% of the smallest value. This means that the model will be slightly more skewed towards the patterns seen in Canberra and Sydney, compared to the rest of the locations, with three exceptions. The last three locations only have 1578 observations. This is about half of the other 46 locations. These three location will therefore have less of an impact in the model training. It is easier for their patterns to be overshadowed by the other locations which have more instances.


### Multivariate Analysis

In [ ]:
# Correlation heatmap.

# Computes the pairwise correlation between each feature.
correlation = encoded_eda_dataset.drop("RainTomorrow", axis=1).corr()

# Shows the correlation between the features in a heatmap.
plt.figure(figsize=(20, 20))
sns.heatmap(correlation, cmap='coolwarm', annot=True, fmt='.2f', vmin=-1, vmax=1)


#### Results


In [ ]:
# Correlation heatmap part 2.

# Computes the pairwise correlation between each feature and the target value.
correlation = encoded_eda_dataset.corr()

# Shows the correlation between the target value and features ordered from the most positive correlation to the most negative correlation.
plt.figure(figsize=(5, 10))
sns.heatmap(correlation[['RainTomorrow']].sort_values(by=['RainTomorrow'],ascending=False), cmap='coolwarm', annot=True, fmt='.2f', vmin=-1, vmax=1)

#### Scatterplot
Some of the most correlated features are investigated further. Additionally, two sets of uncorrelated features are looked at. These scatterplots show correlation if the datapoints follow a straight line with a rate of increase different from zero.

In [ ]:
# Scatterplot.
fig, ax = plt.subplots(3, 2, figsize=(8,25))

ax[0, 0].scatter(encoded_eda_dataset['Temp3pm'], encoded_eda_dataset['Temp9am'])
ax[0, 1].scatter(encoded_eda_dataset['Temp3pm'], encoded_eda_dataset['MaxTemp'])
ax[1, 0].scatter(encoded_eda_dataset['Humidity3pm'], encoded_eda_dataset['Temp3pm'])
ax[1, 1].scatter(encoded_eda_dataset['Humidity9am'], encoded_eda_dataset['Evaporation'])
ax[2, 0].scatter(encoded_eda_dataset['Rainfall'], encoded_eda_dataset['MaxTemp'])
ax[2, 1].scatter(encoded_eda_dataset['RainToday'], encoded_eda_dataset['Sunshine'])

#### Results

#### Pairplot
For a larger overview of the correlation between features, a pairplot is provided to see most features' correlation. To keep the figure readable, both in terms of size of the overall pair plot and the readability of the individual graphs, the most discrete columns are excluded. This includes the target value. Additionally, where there is a value for the morning and afternoon, only the afternoon value is represented in the plot.

In [ ]:
sns.pairplot(encoded_eda_dataset, vars=["Location", "MinTemp", "MaxTemp", "Rainfall", "Sunshine", "Evaporation", "WindGustSpeed", "Pressure3pm", "Humidity3pm"])

#### Results

#### Crosstables
Six crosstables are provided as an alternative to scatterplots for discrete data with few possible values.
The first four, marked with green mapping, is between features. The last two are between a feature and the target value.

In [ ]:
# Crosstables.
fig, ax = plt.subplots(4, 2, figsize=(10,20))

# Create the crosstable objects
xtable1 = pd.crosstab(encoded_eda_dataset['WindGustDir_Sin'],
                      encoded_eda_dataset['Rainfall'], normalize=True)
xtable2 = pd.crosstab(encoded_eda_dataset['Cloud9am'],
                      encoded_eda_dataset['RainToday'], normalize=True)
xtable3 = pd.crosstab(encoded_eda_dataset['WindDir3pm_Sin'],
                      encoded_eda_dataset['WindDir9am_Sin'], normalize=True)
xtable4 = pd.crosstab(encoded_eda_dataset['Cloud9am'],
                      encoded_eda_dataset['Cloud3pm'], normalize=True)


# There are a lot of days in a year, so they need to be grouped into smaller bins for
# the table to be readable.
bins = [50, 100, 150, 200, 250, 300, 350, 366]
days_grouped = pd.cut(encoded_eda_dataset['Day_Sin'], bins)


xtable5 = pd.crosstab(days_grouped,
                      encoded_eda_dataset['Month_Sin'], normalize=True)
# Reverse the grade list, to make a more readable graph
# (Puts the best grade on the top rather than bottom).
# xtable6 = pd.crosstab(encoded_eda_dataset['Grades'],
          #            encoded_eda_dataset, normalize="columns").iloc[::-1]
xtable6 = pd.crosstab(encoded_eda_dataset['RainToday'],
                      encoded_eda_dataset['Month_Sin'])
xtable7 = pd.crosstab(encoded_eda_dataset['Humidity3pm'],
                      encoded_eda_dataset['RainTomorrow'], normalize=True)
xtable8 = pd.crosstab(encoded_eda_dataset['Sunshine'],
                      encoded_eda_dataset['RainTomorrow'], normalize=True)


# Show the crosstables as heatmaps.
# Tables between features.
sns.heatmap(xtable1, cmap='Greens', annot=True, fmt='.1%', ax=ax[0, 0])
sns.heatmap(xtable2, cmap='Greens', annot=True, fmt='.1%', ax=ax[0, 1])
sns.heatmap(xtable3, cmap='Greens', annot=True, fmt='.1%', ax=ax[1, 0])
sns.heatmap(xtable4, cmap='Greens', annot=True, fmt='.1%', ax=ax[1, 1])
sns.heatmap(xtable5, cmap='Greens', annot=True, fmt='.1%', ax=ax[2, 0])
sns.heatmap(xtable6, cmap='Greens', annot=True, fmt='.1%', ax=ax[2, 1])

# Tables between a feature and the target.
sns.heatmap(xtable7, cmap='coolwarm', annot=True, fmt='.1%', ax=ax[3, 0])
sns.heatmap(xtable8, cmap='coolwarm', annot=True, fmt='.1%', ax=ax[3, 1])

### Time Series Analysis